<a href="https://colab.research.google.com/github/narpavi-ai/cctp-481-notes/blob/main/notebooks/01-hello-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Hello, Model

**CCTP 481: Building Your First AI Agent · Module 1**

---

## Let's say you own a food truck

You park it in Edmonton's river valley — **Hawrelak Park** on weekends,
**Louise McKinney** when there's something on downtown, the **Old Strathcona
Farmers' Market** on Saturdays. You sell cinnamon buns, saskatoon berry pie,
bison chili and cold brew.

Every morning you answer the same three questions:

> **What's the weather doing? What have I got left? Do I open today?**

Over the next two days you're going to build an agent that answers them for you.

**Today it can answer none of them — but it can already do plenty else.** That's
the point of this lab: find out exactly what a plain language model is good at,
and exactly where it stops. You can't understand what a framework adds until you
know what you started with.

---

### What you'll do here

Send your first instruction to a large language model **from code** instead of
from a chat box, and see the three things a chat box hides from you: the
**messages**, the **roles**, and the fact that the model is **stateless**.

By the end you will have:

1. A working, free API key
2. A model call you wrote yourself
3. A clear answer to *"what is actually different about calling a model in code?"*
4. Two jobs it does well — and one it can't do at all, which is why Lab 2 exists

### What you need

Nothing installed. This runs entirely in your browser. **No prior Python
required** — you can complete this lab by reading and pressing ▶ on each cell
in order.

### How to run a cell

Click the **▶ play button** to the left of a cell, or click into it and press
**Shift + Enter**. Run them **top to bottom** — later cells depend on earlier ones.

⏱️ About 20 minutes.

<p align="center">
<img src="https://raw.githubusercontent.com/narpavi-ai/cctp-481-notes/main/images/lab1.png" alt="A food truck by the Edmonton river valley with a question-mark bubble above it - the model cannot answer yet" width="640">
</p>

## Before you start — 3 minutes of setup

This notebook talks to Google's Gemini models, so it needs a **Google AI Studio
API key**. It is free — no credit card, no billing account.

**If you already have a key**, use it; you do not need a new one.
**If you don't**, make one now — it takes about a minute:

1. Go to **<https://aistudio.google.com/apikey>** and sign in with any Google account.
2. Click **Create API key**. Copy it.

Then load it into this notebook:

3. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret**.
   - Name: `GOOGLE_API_KEY`
   - Value: paste your key
   - Toggle **Notebook access** on.

> **Why the key icon instead of just pasting it in a cell?** Because anything you
> type in a cell gets saved into the notebook file. If you later share the
> notebook, you'd be sharing your key too. Colab Secrets keeps it out of the file.
> This is a real habit worth building, and it costs you ten seconds.

### Step 1 — Install LangChain

This takes about 30 seconds. You'll see a lot of output; that's normal. You only
need to run it once per session.

In [ ]:
%pip install -q -U "langchain[google-genai]>=1.3,<2"

print("✅ Installed. If Colab offers to restart the runtime, you can ignore it here.")

### Step 2 — Load your key

In [ ]:
import os

# Preferred: Colab Secrets (the key icon in the left sidebar).
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ Key loaded from Colab Secrets.")
except Exception:
    # Fallback for non-Colab, or if you haven't set the secret yet.
    # getpass hides what you type and does NOT save it into the notebook.
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google AI Studio API key: ")
    print("✅ Key loaded for this session only.")

### Step 3 — Create the model

**About the model name.** We use `gemini-3.5-flash-lite` because of **quota**,
not cleverness.

| Model | Requests per day, free tier |
|---|---|
| Flash | 20 |
| **Flash Lite** | **500** (and 15 per minute) |

These four labs need roughly **70 requests** — every step an agent takes is
another one. Check your own limits at <https://aistudio.google.com/rate-limit>.

> **Every provider adds and retires models constantly.** If you get a
> `404 NOT_FOUND`, open <https://aistudio.google.com>, see what your account can
> use, and change the one string below. Nothing else in the notebook changes —
> that is the point of a framework.

In [ ]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-3.5-flash-lite"

model = init_chat_model(MODEL)
print(f"✅ Model ready: {MODEL}")

### A small helper — readable answers

Model replies come back as Markdown: headings, bullet points, **bold**. A plain
`print()` shows you the raw asterisks and hashes. This helper renders them
properly, so you can actually read what your agent said.

Run it once. We'll use `show(...)` for the rest of the course.

In [ ]:
from IPython.display import Markdown, display

# One look for everything the model and the tools say, so a student can tell at
# a glance where the notebook stops talking and the model starts.

ANSWER_LIMIT = 1500


def _text_of(message):
    """The readable text of a model message, whatever shape it arrives in.

    Gemini 3.x returns .content as a LIST of blocks, not a string, so the
    obvious str(response.content) prints a Python list with a base64 signature
    inside it. Everything below goes through here.

    Deliberately does NOT touch .text: calling it is deprecated in LangChain 1.x
    and printed a warning above every single answer.
    """
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\n\n".join(p for p in parts if p)
    return str(content)


def _card(body, label=None, icon="", limit=ANSWER_LIMIT):
    """A labelled, indented block. Markdown inside still renders."""
    body = _text_of(body).strip()
    if not body:
        body = "*(nothing came back)*"
    if len(body) > limit:
        body = body[:limit].rstrip() + f"\n\n*… trimmed here — {len(body):,} characters in full*"
    lines = ["> " + line for line in body.splitlines()]
    if label:
        lines = [f"> {icon} **{label}**".replace(">  ", "> "), ">"] + lines
    return "\n".join(lines)


def show(response, label="Model answer"):
    """Display a model response as a readable answer card."""
    display(Markdown(_card(response, label, icon="🤖")))


def show_text(text, label=None, icon="📄"):
    """Display plain text - a tool result, a lookup - in the same card."""
    display(Markdown(_card(text, label, icon=icon, limit=2500)))


def show_trace(result, label="Agent trace"):
    """Render every step the agent took, in order, as readable cards."""
    msgs = result["messages"]
    out = [f"#### 🔍 {label} — {len(msgs)} steps", ""]

    for i, m in enumerate(msgs, 1):
        kind = type(m).__name__.replace("Message", "").upper()

        if kind == "HUMAN":
            out += [_card(m, f"{i} · You asked", icon="👤", limit=600), ""]

        elif kind == "TOOL":
            name = getattr(m, "name", "tool")
            out += [_card(m, f"{i} · Tool returned — {name}", icon="🛠️", limit=800), ""]

        elif kind == "AI":
            calls = getattr(m, "tool_calls", None)
            if calls:
                steps = []
                for tc in calls:
                    args = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                    steps.append(f"`{tc['name']}({args})`")
                said = _text_of(m).strip()
                body = "\n\n".join(steps + ([said] if said else []))
                out += [_card(body, f"{i} · Model called a tool", icon="🔧", limit=600), ""]
            else:
                out += [_card(m, f"{i} · Model answered", icon="🤖", limit=900), ""]

        else:
            out += [_card(m, f"{i} · {kind}", limit=600), ""]

    display(Markdown("\n".join(out)))


print("✅ Display helpers ready — use show() instead of print() from here on.")


### Step 4 — Your first call

This is the whole thing. One line.

In [ ]:
response = model.invoke(
    "Explain what an AI agent is, in two sentences, to someone who runs a food truck."
)

show(response)

**🎯 Checkpoint.** You just called a large language model from code. If you got
text back, everything downstream in this course will work.

Notice that what came back is **not** a string — it's a response *object*.
`response.content` is the text part of it. There's more in there:

In [ ]:
show_text(f"Python type: {type(response).__name__}", "What came back")
show_text(response.usage_metadata, "Token usage - what this call cost you")

That `usage_metadata` is your first taste of something a chat box never shows
you: **agents cost money per token**, and in Module 4 you'll care a lot about how
many tokens a runaway agent burns before anyone notices.

### Step 5a — Messages and roles

A chat box makes it look like you're "talking to" the model. You aren't. Every
call sends a **list of messages**, each tagged with a **role**. Three of them
matter, and you'll use them constantly:

| Role | Who it represents | What it's for |
|---|---|---|
| `system` | **You, the developer** | Standing instructions: who the model should be, what rules it follows. This is the **system prompt**. |
| `user` | The person using your app | The actual question. This is the **user prompt**. |
| `assistant` | The model | What it said back |

In CCTP 480 you wrote prompts by typing them into a chat window. The
**system prompt** is that same skill — except now it's a parameter in your code
that applies to *every* conversation your app ever has, instead of something you
retype each time.

Let's give the model a job:

In [ ]:
question = "Should I use an AI agent to take customer orders at my food truck?"

cautious = model.invoke([
    {"role": "system", "content":
     "You are a cautious operations manager. Be blunt about what could go wrong."},
    {"role": "user", "content": question},
])

show(cautious, "Answer 1 — the cautious operations manager")

### Step 5b — Change **only** the system prompt

Same question. Same model. Same code. The only thing different is one sentence
in the `system` message — watch what happens to the answer.

In [ ]:
optimist = model.invoke([
    {"role": "system", "content":
     "You are an enthusiastic small-business coach. Focus on the opportunity."},
    {"role": "user", "content": question},   # ← identical question
])

show(optimist, "Answer 2 — the enthusiastic business coach")

**💡 What just happened.** The question never changed. The **system prompt**
changed, and the answer changed completely — tone, priorities, what it chose to
warn you about.

This is the single most useful lever you have over an agent's behaviour, and
you'll set it on every agent you build for the rest of this course.

**It is also a lever with limits.** Hold on to that thought — in Module 4 you'll
try to talk an agent out of a rule you wrote in its system prompt, and discover
that *a rule in the prompt is a request, not a guarantee.*

### Step 6 — The model has no memory

This surprises nearly everyone. Run this:

In [ ]:
model.invoke("My name is Sam and I run a food truck in Edmonton's river valley.")

second = model.invoke("What is my name?")
show(second)

**It doesn't know.** Each `invoke()` is a completely fresh request. The model kept
nothing — not your name, not your business, not a single word.

The only reason ChatGPT *seems* to remember is that the app re-sends the whole
conversation every single time. Here's that, by hand:

In [ ]:
conversation = [
    {"role": "user", "content": "My name is Sam and I run a food truck in Edmonton's river valley."},
    {"role": "assistant", "content": "Nice to meet you, Sam!"},
    {"role": "user", "content": "What is my name?"},
]

show(model.invoke(conversation))

**💡 Why this matters for the rest of the course.** Memory is not something the
model *has* — it's something *you build*. In Module 3 you'll stop hand-managing
lists like the one above and let LangGraph do it. But it's doing exactly this
underneath, and now you've seen it.

### Step 7 — Put it to work

You have a working model. Before you find its limits, use it for what it is
genuinely, immediately good at — because this alone is worth the price of
admission for a small business.

Two real jobs off your actual to-do list.

In [ ]:
show(model.invoke(
    "Write a friendly two-sentence description of our cinnamon buns "
    "for the food truck's menu board. Warm, not corporate."
), "Job 1 — write my menu copy")

In [ ]:
show(model.invoke(
    "A customer emailed to say the bison chili they bought yesterday was cold "
    "by the time they got home. Draft a short, warm reply that apologises, "
    "offers a refund or a replacement, and doesn't sound like a form letter."
), "Job 2 — draft a reply to an unhappy customer")

**🎯 Checkpoint.** Both of those are good. Not perfect — you'd tweak them — but
they are a genuinely useful first draft, in about two seconds, for jobs that
would otherwise eat twenty minutes of your evening.

**This is what a language model is for.** Writing, rephrasing, summarising,
drafting, changing tone. It is very good at anything where the raw material is
already in the question.

Now let's find the edge of that.

### Step 8 — The one it can't do

Same model. Same code. A question you'd ask on any Saturday morning.

In [ ]:
show(model.invoke(
    "What's the temperature in Edmonton right now, "
    "and how many cinnamon buns do I have left in the truck?"
), "The Saturday morning question")

### 🚧 Two gaps, and they are the rest of this course

It couldn't answer either half — and for **two different reasons**, which is the
thing worth carrying out of this lab:

| | The gap | Why |
|---|---|---|
| 🌍 | **It can't see the world** | No live data. Nothing that changed after it finished training — no weather, no traffic, no prices. |
| 🔒 | **It can't see your business** | Your inventory, your prices, your handbook. None of it was in its training data, and it has no way to look. |

Notice it didn't *pretend*. It told you it couldn't. That's the model behaving
well — and it is also exactly why a model alone can't run your morning.

> **This is what a tool is.** A tool is how you close one of those gaps: you hand
> the model a function it can call when it needs a fact it doesn't have.
>
> **In Lab 2 you close both.** You'll give this same model a live weather lookup
> and a stock lookup, ask this same question, and get a real answer — because it
> decides, on its own, which tool to call.
>
> **That decision is what makes it an agent.**

### Optional — the model can see, too

`gemini-3.5-flash-lite` is **multimodal**: it takes images as well as text. If you've
got a photo handy — a handwritten prep list, a whiteboard, a receipt, a shelf of
supplies — try it.

Run the cell, pick a file when prompted. **Skip this if you're short on time**;
nothing later depends on it.

In [ ]:
# Optional. Upload any image and ask the model about it.
from google.colab import files
import base64

uploaded = files.upload()          # pick a photo of a list, a receipt, a shelf...

if uploaded:
    name = list(uploaded)[0]
    b64 = base64.b64encode(uploaded[name]).decode()

    reply = model.invoke([{"role": "user", "content": [
        {"type": "text", "text":
         "You are helping a food truck owner. Read this image and list "
         "anything in it that looks like an inventory item or a to-do."},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
    ]}])

    show(reply, f"What the model sees in {name}")

**💡 Worth noticing.** Reading a photo is still just `model.invoke(...)` — the
same call, with a different kind of content. But it's *still* not an agent: it
answered from what you gave it, and it still couldn't check a single fact.

### Your turn

Two small experiments. There's no autograder — the point is to build intuition.

1. **Write a system prompt for your own work.** Not the food truck — your actual
   job. *"You are an assistant for a [your role]. Always [some rule that matters
   in your work]."* Then ask it something you'd genuinely ask. Change only the
   system prompt and ask the same question again.
2. **Break it deliberately.** Change `MODEL` to `"google_genai:not-a-real-model"`
   and run it. Read the error. Recognising errors on sight is worth more than
   avoiding them — and this is the exact error you'll hit for real the next time
   Google retires a model.

In [ ]:
# Your experiment here.

my_system = "You are an assistant for a ...  Always ..."
my_question = "..."

# show(model.invoke([
#     {"role": "system", "content": my_system},
#     {"role": "user", "content": my_question},
# ]))

## If something breaks

These are the errors this notebook actually produces. You're not doing anything
wrong — everyone hits at least one.

| What you see | What it means | Fix |
|---|---|---|
| `404 NOT_FOUND` / `GoogleModelNotFoundError` / *"no longer available to new users"* | **The most likely error in this notebook.** Google retired that model. | Open <https://aistudio.google.com>, check which models you can use, edit the `MODEL` string. Nothing else changes. |
| `MODEL_AUTHENTICATION` / 401 / `API key not valid` | The key didn't load | Re-run the setup cell. Check the secret is named exactly `GOOGLE_API_KEY` and Notebook access is on |
| `429` / `RESOURCE_EXHAUSTED` | Too many requests too fast on the free tier | Wait 60 seconds and re-run. Don't re-run in a loop |
| `ModuleNotFoundError` | The install cell didn't run, or the runtime restarted | Re-run the install cell at the top, then re-run from there |
| `NameError: show is not defined` | You skipped the helper cell | Run the `show()` cell in Step 3 |

Still stuck? Flag it in chat — an error on screen is a better teaching moment
than a working cell.

---

### What you learned

- Calling a model from code returns an **object**, not just text — including token usage
- Every call is a list of **messages with roles**; the **system prompt** is your main control
- **The model is stateless.** Memory is something you build, and Module 3 is where you build it
- A plain model **can't see the world and can't see your business** — and closing those
  two gaps is what the rest of this course is about

### References

Go deeper on anything in this lab:

**LangChain docs**

- [Quickstart](https://docs.langchain.com/oss/python/langchain/quickstart) — the official version of this lab
- [Models](https://docs.langchain.com/oss/python/langchain/models) — `init_chat_model`, providers, and swapping between them
- [Messages](https://docs.langchain.com/oss/python/langchain/messages) — roles, content blocks, and what a response object holds
- [Install LangChain](https://docs.langchain.com/oss/python/langchain/install)
- [LangChain overview](https://docs.langchain.com/oss/python/langchain/overview) and [Philosophy](https://docs.langchain.com/oss/python/langchain/philosophy) — why the framework exists at all

**Getting a key / free tier**

- [Google AI Studio](https://aistudio.google.com/apikey) — create your free API key
- [Gemini models](https://ai.google.dev/gemini-api/docs/models) — what exists today, and what got retired
- [Gemini API pricing](https://ai.google.dev/gemini-api/docs/pricing) — what the free tier covers, and the note that free-tier content is used to improve Google's products
- [Gemini rate limits](https://ai.google.dev/gemini-api/docs/rate-limits) — Google no longer publishes a fixed table; your own limits are shown in AI Studio

**If you'd rather not use Google:** [Groq](https://console.groq.com) and
[Cerebras](https://cloud.cerebras.ai) also issue free keys without a credit card.
Change the `MODEL` string and install the matching package — the rest of the
notebook is unchanged.

### The no-code version of this lab

Everything above exists as an n8n workflow: **[`n8n/01-hello-model.json`](n8n/01-hello-model.json)**.
Same model, same failure, no Python. Setup steps are in
**[`n8n/SETUP.md`](n8n/SETUP.md)**.

### Next

**Lab 2 — Build Your First Agent.** You have a model that writes well and knows
nothing about today. Next you give it **tools** — a live weather lookup and a
stock lookup — and the question it just refused becomes one it answers.